<a href="https://colab.research.google.com/github/yianli0213/programming-language/blob/main/%E3%80%8CHW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%842_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install gspread gspread_dataframe google-auth google-auth-oauthlib google-auth-httplib2 \
               gradio pandas beautifulsoup4 google-generativeai python-dateutil

In [20]:
from urllib.parse import urljoin
import os, time, uuid, re, json, datetime
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup

import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.auth import default

In [11]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [6]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('tomato')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-pro')

In [7]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1xc2xqhsNEYPbbJUEqKOf5tJ8Zc6t0mavpNPSdDV6GW0/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"
TIMEZONE = "Asia/Taipei"

In [8]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url(SHEET_URL)


# 從 gsheets 的 All-whiteboard-device 載入 sheets
sh = gsheets.worksheet(WORKSHEET_NAME).get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sh[1:], columns=sh[0])
# 取得最前面的5筆資料
df.head()


""


In [12]:
# Google Sheet 連線（修正版）
gsheets = gc.open_by_url(SHEET_URL)

def ensure_worksheet(spreadsheet, title, header):
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows='1000', cols=str(len(header)+5))
        ws.update([header])
    data = ws.get_all_values()
    if not data:
        ws.update([header])
    elif data[0] != header:
        ws.clear()
        ws.update([header])
    return ws

TASKS_HEADER = [
    'id','task','status','priority','est_min','start_time','end_time',
    'actual_min','pomodoros','due_date','labels','notes',
    'created_at','updated_at','completed_at','planned_for'
]
LOGS_HEADER  = ['log_id','task_id','phase','start_ts','end_ts','minutes','cycles','note']
CLIPS_HEADER = ['clip_id','url','selector','text','href','created_at','added_to_task']

ws_tasks = ensure_worksheet(gsheets, WORKSHEET_NAME, TASKS_HEADER)
ws_logs  = ensure_worksheet(gsheets, 'pomodoro_logs', LOGS_HEADER)
ws_clips = ensure_worksheet(gsheets, 'web_clips', CLIPS_HEADER)
print('✅ 工作表連線完成')

✅ 工作表連線完成


In [13]:
def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    data = ws.get_all_values()
    if not data or len(data) <= 1:
        return pd.DataFrame(columns=header)
    df = pd.DataFrame(data[1:], columns=data[0]).fillna('')
    for c in header:
        if c not in df.columns:
            df[c] = ''
    for col in ['est_min','actual_min','pomodoros']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    return df[header]

def write_df(ws, df, header):
    ws.clear()
    if df.empty:
        ws.update([header])
        return
    df_out = df.copy()
    for c in header:
        if c not in df_out.columns:
            df_out[c] = ''
    df_out = df_out[header].fillna('')
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)
    ws.update([header] + df_out.values.tolist())

def refresh_all():
    return (
        read_df(ws_tasks, TASKS_HEADER).copy(),
        read_df(ws_logs,  LOGS_HEADER).copy(),
        read_df(ws_clips, CLIPS_HEADER).copy()
    )

tasks_df, logs_df, clips_df = refresh_all()
print('✅ 資料載入完成')

✅ 資料載入完成


In [14]:
def list_task_choices():
    global tasks_df
    if tasks_df.empty:
        return []
    df = tasks_df[tasks_df['status'] != 'done'].copy()
    if df.empty:
        return []
    def row_label(r):
        return f"[{r['status']}] (P:{r['priority']}) {r['task']}"
    return [(row_label(r), r['id']) for _, r in df.iterrows()]

def get_task_cycles(task_id):
    global tasks_df
    if not task_id:
        return gr.update(value=1)
    row = tasks_df[tasks_df['id'] == task_id]
    if row.empty:
        return gr.update(value=1)
    est_min = int(row.iloc[0]['est_min'])
    cycles = max(1, int((est_min + 24) // 25))
    return gr.update(value=cycles)

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df
    if not task or not str(task).strip():
        return '⚠️ 請輸入任務名稱', tasks_df, gr.update(), gr.update()
    _now = tznow().isoformat()
    task_id = str(uuid.uuid4())[:8]
    new = pd.DataFrame([{
        'id': task_id, 'task': str(task).strip(), 'status': 'todo',
        'priority': priority or 'M',
        'est_min': int(est_min) if est_min else 25,
        'start_time': '', 'end_time': '', 'actual_min': 0, 'pomodoros': 0,
        'due_date': due_date or '', 'labels': labels or '', 'notes': notes or '',
        'created_at': _now, 'updated_at': _now, 'completed_at': '',
        'planned_for': planned_for or ''
    }])
    tasks_df = pd.concat([tasks_df, new], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    choices = list_task_choices()
    return f'✅ 已新增：{str(task).strip()}', tasks_df, gr.update(choices=choices, value=task_id), gr.update(choices=choices, value=task_id)

def update_task_status(task_id, new_status):
    global tasks_df
    if not task_id:
        return '⚠️ 請先選擇任務', tasks_df, gr.update(), gr.update()
    idx = tasks_df.index[tasks_df['id'] == task_id]
    if len(idx) == 0:
        return '⚠️ 找不到任務', tasks_df, gr.update(), gr.update()
    i = idx[0]
    tasks_df.loc[i, 'status'] = new_status
    tasks_df.loc[i, 'updated_at'] = tznow().isoformat()
    if new_status == 'done':
        tasks_df.loc[i, 'completed_at'] = tznow().isoformat()
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    choices = list_task_choices()
    return '✅ 狀態已更新', tasks_df, gr.update(choices=choices, value=None), gr.update(choices=choices, value=None)

def mark_done(task_id):
    return update_task_status(task_id, 'done')

print('✅ 任務函數載入完成')

✅ 任務函數載入完成


In [16]:
def recalc_task_actuals(task_id):
    global tasks_df, logs_df
    work_logs = logs_df[(logs_df['task_id']==task_id) & (logs_df['phase']=='work')]
    total_min = work_logs['minutes'].astype(float).sum() if not work_logs.empty else 0
    pomos = int(total_min // 25)
    idx = tasks_df.index[tasks_df['id']==task_id]
    if len(idx) == 0: return
    i = idx[0]
    tasks_df.loc[i, 'actual_min'] = int(total_min)
    tasks_df.loc[i, 'pomodoros'] = pomos
    tasks_df.loc[i, 'updated_at'] = tznow().isoformat()

_active_sessions = {}

def format_seconds(seconds):
    seconds = max(0, int(seconds))
    m = seconds // 60
    s = seconds % 60
    return f'{m:02d}:{s:02d}'

def update_timer_display(task_id):
    if not task_id or task_id not in _active_sessions:
        return '⏱️ 尚未開始計時'
    sess = _active_sessions[task_id]
    start = pd.to_datetime(sess['start_ts'])
    elapsed = (tznow() - start).total_seconds()
    phase = sess['phase']
    cycles = int(sess['cycles'])
    target = (25 * cycles * 60) if phase == 'work' else (15 * 60)
    remaining = target - elapsed
    phase_name = '工作' if phase == 'work' else '休息'
    return (
        f'⏱️ **{phase_name}中**\n\n'
        f'已計時：{format_seconds(elapsed)}\n\n'
        f'剩餘：{format_seconds(remaining)}'
    )

def start_phase(task_id, phase, cycles):
    if not task_id: return '⚠️ 請先選擇任務'
    _active_sessions[task_id] = {
        'phase': phase,
        'start_ts': tznow().isoformat(),
        'cycles': int(cycles) if cycles else 1
    }
    label = '工作（每25分鐘一番茄）' if phase == 'work' else '休息'
    return f'▶️ 開始{label}'

def end_phase(task_id, note):
    global logs_df, tasks_df
    if not task_id: return '⚠️ 請先選擇任務'
    if task_id not in _active_sessions: return '⚠️ 尚未開始任何階段'
    sess = _active_sessions.pop(task_id)
    start = pd.to_datetime(sess['start_ts'])
    end = tznow()
    minutes = round((end - start).total_seconds() / 60.0, 2)
    log = pd.DataFrame([{
        'log_id': str(uuid.uuid4())[:8],
        'task_id': task_id, 'phase': sess['phase'],
        'start_ts': start.isoformat(), 'end_ts': end.isoformat(),
        'minutes': minutes, 'cycles': int(sess['cycles']), 'note': note or ''
    }])
    logs_df = pd.concat([logs_df, log], ignore_index=True)
    write_df(ws_logs, logs_df, LOGS_HEADER)
    if sess['phase'] == 'work':
        recalc_task_actuals(task_id)
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return f'⏹️ 已結束 {sess["phase"]}，紀錄 {minutes} 分鐘'

print('✅ 番茄鐘函數載入完成')

✅ 番茄鐘函數載入完成


In [17]:
def generate_today_plan():
    global tasks_df
    today = tznow().date().isoformat()
    cand = tasks_df[
        ((tasks_df['due_date']==today) | (tasks_df['planned_for'].str.lower()=='today')) &
        (tasks_df['status']!='done')
    ].copy()
    if cand.empty:
        return '📭 今天沒有標記的任務。請把 due_date 設為今天，或 planned_for 設為 today。'

    pr_order = {'H':0,'M':1,'L':2}
    cand['p_ord'] = cand['priority'].map(pr_order).fillna(3)
    cand = cand.sort_values(['p_ord','est_min'], ascending=[True,True])

    items = []
    for _, r in cand.iterrows():
        items.append({'id': r['id'], 'task': r['task'],
                      'est_min': int(r['est_min']), 'priority': r['priority']})

    sys_prompt = (
        '你是一位任務規劃助理。請把輸入的任務排成三段：morning、afternoon、evening，'
        '並給出每段的重點與時間預估。回傳 Markdown 格式：\n'
        '### Morning\n- [任務ID] 任務名稱（預估 xx 分）\n'
        '### Afternoon\n...\n### Evening\n...\n'
    )
    user_content = json.dumps({'today': today, 'tasks': items}, ensure_ascii=False)

    try:
        resp = model.generate_content(sys_prompt + '\n\n' + user_content)
        return resp.text
    except Exception as e:
        buckets = {'morning':[], 'afternoon':[], 'evening':[]}
        for i, (_, r) in enumerate(cand.iterrows()):
            list(buckets.values())[i % 3].append(r)
        def sec_md(name, rows):
            if not rows: return f'### {name.title()}\n（無）\n'
            lines = [f'### {name.title()}']
            for r in rows:
                lines.append(f"- [{r['id']}] {r['task']}（預估 {int(r['est_min'])} 分，P:{r['priority']}）")
            return '\n'.join(lines) + '\n'
        fallback = '\n'.join([sec_md(k, v) for k, v in buckets.items()])
        return f'⚠️ Gemini 失敗（{e}），規則式規劃：\n\n{fallback}'

def today_summary():
    global tasks_df
    today = tznow().date().isoformat()
    planned = tasks_df[
        (tasks_df['due_date']==today) | (tasks_df['planned_for'].str.lower()=='today')
    ]
    done = planned[planned['status']=='done']
    total = len(planned)
    done_n = len(done)
    rate = (done_n/total*100) if total > 0 else 0
    return f'📅 今日計畫：{total}　✅ 完成：{done_n}　📈 完成率：{rate:.1f}%'

print('✅ AI 計畫函數載入完成')

✅ AI 計畫函數載入完成


In [18]:
def crawl(url, selector, keyword, mode, limit):
    if not url or not str(url).strip():
        return pd.DataFrame(columns=CLIPS_HEADER), '⚠️ 請輸入目標 URL'
    try:
        resp = requests.get(url, timeout=15, headers={'User-Agent':'Mozilla/5.0'})
        resp.raise_for_status()
    except requests.exceptions.HTTPError as e:
        return pd.DataFrame(columns=CLIPS_HEADER), f'⚠️ 網站拒絕存取：{e}'
    except requests.exceptions.Timeout:
        return pd.DataFrame(columns=CLIPS_HEADER), '⚠️ 連線逾時'
    except requests.exceptions.RequestException as e:
        return pd.DataFrame(columns=CLIPS_HEADER), f'⚠️ 連線失敗：{e}'

    soup = BeautifulSoup(resp.text, 'html.parser')
    sel = selector.strip() if selector and selector.strip() else 'a'
    try:
        nodes = soup.select(sel)
    except Exception as e:
        return pd.DataFrame(columns=CLIPS_HEADER), f'⚠️ CSS Selector 錯誤：{e}'

    rows = []
    kw = keyword.strip().lower() if keyword and keyword.strip() else ''

    for n in nodes:
        text = re.sub(r'\s+', ' ', n.get_text(' ', strip=True)).strip()
        href = ''
        tag = n if n.name == 'a' else n.find('a')
        if tag:
            href = urljoin(url, tag.get('href', ''))
        if not text and not href: continue
        if kw and kw not in text.lower() and kw not in href.lower(): continue
        rows.append({
            'clip_id': str(uuid.uuid4())[:8],
            'url': url, 'selector': sel,
            'text': text, 'href': href,
            'created_at': tznow().isoformat(),
            'added_to_task': ''
        })
        if limit and len(rows) >= int(limit): break

    df = pd.DataFrame(rows, columns=CLIPS_HEADER)
    if df.empty:
        return df, '⚠️ 沒有抓到資料'
    return df, f'✅ 成功擷取 {len(df)} 筆'

def add_clips_as_tasks(clip_ids, default_priority, est_min):
    global clips_df, tasks_df
    if not clip_ids:
        return '⚠️ 請輸入 clip_id', clips_df, tasks_df, gr.update(), gr.update()
    sel = clips_df[clips_df['clip_id'].isin(clip_ids)]
    if sel.empty:
        return '⚠️ 找不到對應 clip_id', clips_df, tasks_df, gr.update(), gr.update()
    _now = tznow().isoformat()
    new_tasks = []
    for _, r in sel.iterrows():
        title = r['text'] or r['href'] or '（未命名）'
        new_tasks.append({
            'id': str(uuid.uuid4())[:8], 'task': title[:120], 'status': 'todo',
            'priority': default_priority or 'M',
            'est_min': int(est_min) if est_min else 25,
            'start_time':'','end_time':'','actual_min':0,'pomodoros':0,
            'due_date':'', 'labels':'from:crawler',
            'notes': f"來源：{r['url']}\n連結：{r['href']}",
            'created_at':_now,'updated_at':_now,'completed_at':'','planned_for':''
        })
    tasks_df = pd.concat([tasks_df, pd.DataFrame(new_tasks)], ignore_index=True)
    clips_df.loc[clips_df['clip_id'].isin(clip_ids), 'added_to_task'] = 'yes'
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    write_df(ws_clips, clips_df, CLIPS_HEADER)
    choices = list_task_choices()
    return f'✅ 已加入 {len(new_tasks)} 項任務', clips_df, tasks_df, gr.update(choices=choices), gr.update(choices=choices)

def _refresh():
    global tasks_df, logs_df, clips_df
    tasks_df, logs_df, clips_df = refresh_all()
    choices = list_task_choices()
    return tasks_df, logs_df, clips_df, gr.update(choices=choices), gr.update(choices=choices), today_summary()

print('✅ 爬蟲函數載入完成')

✅ 爬蟲函數載入完成


In [21]:
with gr.Blocks(title='HW3 待辦清單與番茄鐘') as demo:
    gr.Markdown('# ✅ 待辦清單與番茄鐘（Google Sheet 同步版）')
    with gr.Row():
        btn_refresh = gr.Button('🔄 重新整理')
        out_summary = gr.Markdown(today_summary())

    with gr.Tab('Tasks'):
        with gr.Row():
            with gr.Column(scale=2):
                inp_task     = gr.Textbox(label='任務名稱')
                inp_priority = gr.Dropdown(['H','M','L'], value='M', label='優先級')
                inp_est      = gr.Number(value=25, label='預估時間（分鐘）', precision=0)
                inp_due      = gr.Textbox(label='到期日（YYYY-MM-DD）')
                inp_labels   = gr.Textbox(label='標籤')
                inp_notes    = gr.Textbox(label='備註')
                inp_planned  = gr.Dropdown(['','today','tomorrow'], value='', label='規劃歸屬')
                btn_add      = gr.Button('➕ 新增任務')
                msg_add      = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df, label='任務清單', interactive=False)
        with gr.Row():
            task_choice = gr.Dropdown(choices=list_task_choices(), label='選取任務', value=None)
            new_status  = gr.Dropdown(['todo','in-progress','done'], value='in-progress', label='更新狀態')
            btn_update  = gr.Button('✏️ 更新狀態')
            btn_done    = gr.Button('✅ 標記完成')
            msg_update  = gr.Markdown()

    with gr.Tab('Pomodoro'):
        with gr.Row():
            sel_task = gr.Dropdown(choices=list_task_choices(), label='選擇任務', value=None)
            cycles   = gr.Number(value=1, precision=0, label='番茄數（自動計算）')
        timer_display = gr.Markdown('⏱️ 尚未開始計時')
        timer = gr.Timer(1)
        with gr.Row():
            btn_start_work = gr.Button('▶️ 開始工作')
            note_work      = gr.Textbox(label='工作備註')
            btn_end_work   = gr.Button('⏹️ 結束工作並記錄')
        with gr.Row():
            btn_start_break = gr.Button('🍵 開始休息')
            note_break      = gr.Textbox(label='休息備註')
            btn_end_break   = gr.Button('⏹️ 結束休息並記錄')
        msg_pomo  = gr.Markdown()
        grid_logs = gr.Dataframe(value=logs_df, label='番茄鐘紀錄', interactive=False)

    with gr.Tab('AI Plan'):
        gr.Markdown('根據今天的任務用 Gemini 排行程。')
        btn_plan = gr.Button('🧠 產生今日計畫')
        out_plan = gr.Markdown()

    with gr.Tab('Crawler'):
        inp_url      = gr.Textbox(label='目標 URL')
        inp_selector = gr.Textbox(label='CSS Selector（空白預設抓 a 標籤）')
        inp_keyword  = gr.Textbox(label='關鍵字篩選（空白＝全部保留）')
        inp_mode     = gr.Radio(['text','href','both'], value='both', label='擷取內容')
        inp_limit    = gr.Number(value=20, precision=0, label='最多幾筆')
        btn_crawl    = gr.Button('🕷️ 開始擷取')
        msg_crawl    = gr.Markdown()
        grid_clips   = gr.Dataframe(value=clips_df, label='擷取結果', interactive=True)
        inp_clip_ids      = gr.Textbox(label='要加入任務的 clip_id（逗號分隔）')
        inp_clip_priority = gr.Dropdown(['H','M','L'], value='L', label='優先級')
        inp_clip_est      = gr.Number(value=25, precision=0, label='預估分鐘')
        btn_add_clips     = gr.Button('➕ 加入為任務')
        msg_add_clips     = gr.Markdown()

    with gr.Tab('Summary'):
        btn_summary  = gr.Button('📊 計算今日完成率')
        out_summary2 = gr.Markdown()

    btn_refresh.click(_refresh, outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary])

    btn_add.click(
        add_task,
        inputs=[inp_task, inp_priority, inp_est, inp_due, inp_labels, inp_notes, inp_planned],
        outputs=[msg_add, grid_tasks, task_choice, sel_task]
    )
    btn_update.click(update_task_status, inputs=[task_choice, new_status],
                     outputs=[msg_update, grid_tasks, task_choice, sel_task])
    btn_done.click(mark_done, inputs=[task_choice],
                   outputs=[msg_update, grid_tasks, task_choice, sel_task])

    sel_task.change(get_task_cycles, inputs=[sel_task], outputs=[cycles])
    timer.tick(update_timer_display, inputs=[sel_task], outputs=[timer_display])

    btn_start_work.click(start_phase, inputs=[sel_task, gr.State('work'), cycles], outputs=[msg_pomo])
    btn_end_work.click(end_phase, inputs=[sel_task, note_work], outputs=[msg_pomo])
    btn_start_break.click(start_phase, inputs=[sel_task, gr.State('break'), cycles], outputs=[msg_pomo])
    btn_end_break.click(end_phase, inputs=[sel_task, note_break], outputs=[msg_pomo])

    btn_plan.click(generate_today_plan, outputs=[out_plan])

    def _crawl_and_save(u, s, k, m, l):
        global clips_df
        df, msg = crawl(u, s, k, m, l)
        if not df.empty:
            clips_df = pd.concat([clips_df, df], ignore_index=True)
            write_df(ws_clips, clips_df, CLIPS_HEADER)
        return msg, clips_df

    btn_crawl.click(_crawl_and_save,
                    inputs=[inp_url, inp_selector, inp_keyword, inp_mode, inp_limit],
                    outputs=[msg_crawl, grid_clips])

    def _add_clips(ids_str, pr, est):
        ids = [c.strip() for c in (ids_str or '').split(',') if c.strip()]
        return add_clips_as_tasks(ids, pr, est)

    btn_add_clips.click(_add_clips,
                        inputs=[inp_clip_ids, inp_clip_priority, inp_clip_est],
                        outputs=[msg_add_clips, grid_clips, grid_tasks, task_choice, sel_task])

    btn_summary.click(today_summary, outputs=[out_summary2])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4b59ec8c75d3091801.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_5925/433829005.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out = df_out[header].fillna('')
/tmp/ipykernel_5925/1638120888.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  logs_df = pd.concat([logs_df, log], ignore_index=True)
/tmp/ipykernel_5925/433829005.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://4b59ec8c75d3091801.gradio.live
